# Chess Style Fingerprint Walkthrough

## Project Overview

This project learns to distinguish chess playing styles from PGN game files using deep learning.

**Main goal:** Given a sequence of chess moves, can a neural network predict which player (or engine) made those moves?

**Why this matters:** Chess style is a behavioral signature. Players have recognizable preferences for piece placement, risk-taking, and tactical decisions. This notebook shows how to encode chess games as tensors, train neural models, and make style predictions on unseen games.

## Notebook Structure

This walkthrough progresses through the full ML pipeline:

1. **Data preparation**: Load PGN files and convert chess positions to tensors
2. **Dataset building**: Create train/validation/test splits with proper sampling
3. **Model definition**: Compare two neural architectures (baseline vs. Transformer-based)
4. **Training**: Optimize the model on labeled games
5. **Evaluation**: Test accuracy and error analysis
6. **Inference**: Predict style distribution for a new game

Each cell is independent enough to run on its own, but cells build on each other. If you get errors, make sure you've run all previous cells in order.

## 1. Set Up the Notebook Environment

This cell ensures we're in the right working directory and that Python can import our local `src` modules. It also reloads any modules that might be cached from a previous kernel session, ensuring we always get fresh code.

In [14]:
from pathlib import Path
import importlib
import os
import sys

# Determine where we are and navigate to the project root if needed
cwd = Path.cwd()
if cwd.name != "chess-style-fingerprint" and (cwd / "chess-style-fingerprint").exists():
    os.chdir(cwd / "chess-style-fingerprint")

# Add the project root to sys.path so imports find the local src/ package
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Force reload of any modules that might be cached in the kernel
# This ensures we pick up the latest code from src/
for module_name in ["src.dataset", "src.model", "src.pipeline"]:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

print("Project root:", project_root)
print("src exists:", (project_root / "src").exists())

Project root: /Users/aadil/Chess-ML-Project/chess-style-fingerprint
src exists: True


## 2. Inspect the Data Layout

The training code expects a strict folder structure: `data/raw/<label_name>/*.pgn`. Each folder name becomes a class label (e.g., `data/raw/tal/`, `data/raw/fischer/`, `data/raw/stockfish/`). This cell shows which class folders exist and how many usable games we have in each.

In [ ]:
from collections import Counter
from src.dataset import LABELS, collect_game_records

data_dir = Path("data/raw")

# Scan all PGN files and count how many valid games we can extract
records = collect_game_records(str(data_dir))
game_counts = Counter(record.label_name for record in records)

print("=== Data Layout ===")
print("Folder structure: data/raw/<label_name>/*.pgn")
print()
print("Label ID | Label Name       | PGN Files | Usable Games")
print("-" * 56)

for name, label_id in LABELS.items():
    pgn_count = len(list((data_dir / name).glob("**/*.pgn")))
    pgn_gz_count = len(list((data_dir / name).glob("**/*.pgn.gz")))
    usable_games = game_counts.get(name, 0)

    print(
        f"  {label_id:2d}   | {name:14s} | {pgn_count + pgn_gz_count:9d} | {usable_games:12d}"
    )

Known labels:
   0  tal           PGN files:   1  usable games:  1801
   1  petrosian     PGN files:   1  usable games:  1316
   2  karpov        PGN files:   1  usable games:  2964
   3  fischer       PGN files:   1  usable games:   729
   4  kasparov      PGN files:   1  usable games:  1820
   5  carlsen       PGN files:   1  usable games:  7046
   6  anand         PGN files:   1  usable games:  3626
   7  botvinnik     PGN files:   1  usable games:   774
   8  stockfish     PGN files:   1  usable games:   998


## 3. Convert a Board to a Tensor

Chess is a spatial game played on an 8×8 board. To feed board states into a neural network, we encode them as multi-channel images.

### The 34-Channel Move Representation

Instead of encoding just the board position, we encode *moves*. Each move is represented as:

- **Channels 0–11** (12 piece planes): Position BEFORE the move
  - 6 planes for White pieces (pawn, knight, bishop, rook, queen, king)
  - 6 planes for Black pieces

- **Channels 12–23** (12 piece planes): Position AFTER the move
  - Same layout: White pieces (0–5), Black pieces (6–11)

- **Channels 24–33** (10 metadata planes):
  - Repetition status, castling rights, side-to-move, move counters, board-edge context

**Why encode before and after?** This captures not just the piece configuration, but the *action* that was taken. The move from-square and to-square are implicit in the before/after difference.

### Perspective Normalization

The board is always encoded from the **mover's perspective**. If Black just moved, the board is rotated 180° so Black's pieces appear at the bottom (like the player's home rank). This helps the model learn style-invariant features.

### Spatial Input for CNNs

The resulting `(34, 8, 8)` tensor is ideal for convolutional neural networks:
- 2D spatial structure (8×8 board)
- Multiple channels (piece types, metadata)
- Invariances: CNNs learn translation and rotation-aware features

In [ ]:
import chess
from src.dataset import board_to_tensor

# Start from the standard opening position
board = chess.Board()

# Convert the board to a 34-channel tensor using the encoding described above
x = board_to_tensor(board)

print("Standard starting chess position:")
print(board)
print("\n--- Tensor Encoding ---")
print(f"Shape: {tuple(x.shape)} (channels, height, width)")
print(f"Total values in tensor: {x.numel()}")
print(f"Nonzero entries: {int(x.sum().item())} (one per occupied square)")
print(f"\nNote: In the opening position, both 'before' and 'after' piece planes")
print(f"show the same board (no move has been made yet).")

Board:
r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R

Tensor shape: (14, 8, 8)
Nonzero tensor entries: 76


## 4. Convert a PGN Into Sliding Windows

### Why Windows?

A full chess game has 40–80 moves. Rather than labeling the entire game as one sample, we create many overlapping *windows* of moves. This:

1. **Increases data**: Each game yields multiple training examples
2. **Focuses on middlegame/endgame**: We skip the opening (first 10 moves) to avoid memorizing standard theory
3. **Captures decision patterns**: A short sequence (15 moves) lets the model learn recurring style patterns

### Sliding Window Process

1. Read a PGN file
2. For each game in the file:
   - Play through all mainline moves
   - Collect move tensors after skipping the first `skip_moves` (default: 10)
   - Create windows of size `window_size` (default: 15) with stride `stride` (default: 5)
3. Each window gets labeled with the player/engine class

**Example:** If a game has 60 moves and we skip the first 10, we have 50 move tensors left. With `window_size=15` and `stride=5`, we get windows at positions 0, 5, 10, 15, ... = **9 overlapping windows per game**.

In [ ]:
from src.dataset import pgn_to_windows

# Find all PGN files in the data directory
all_pgns = sorted(data_dir.glob("**/*.pgn"))

if not all_pgns:
    print("No PGN files found yet. Add files under data/raw/<label_name>/ before running this step.")
else:
    # Use the first PGN as an example
    example_pgn = all_pgns[0]
    
    # Extract the player/engine name from the folder path
    label_name = example_pgn.relative_to(data_dir).parts[0].lower()
    label_id = LABELS[label_name]
    
    # Convert the PGN file into windows of move tensors
    # Default: window_size=15, stride=5, skip_moves=10
    samples = pgn_to_windows(str(example_pgn), label=label_id)

    print(f"Example PGN file: {example_pgn}")
    print(f"Detected player/engine: {label_name} (label_id={label_id})")
    print(f"\nWindows extracted from this file: {len(samples)}")
    print(f"Each window will be (15, 34, 8, 8): 15 consecutive move tensors")
    
    if samples:
        window_tensor, y = samples[0]
        print(f"\n--- First Window ---")
        print(f"Shape: {tuple(window_tensor.shape)}")
        print(f"Label: {y}")
        print(f"Interpretation: 15 sequential move states from {label_name}'s game")

Example PGN: data/raw/anand/Anand.pgn
Label: anand 6
Windows extracted: 49503
First window shape: (15, 14, 8, 8)
First window label: 6


## 5. Build Train, Validation, and Test Splits

### Why Stratified Game-Level Splits?

The data pipeline carefully avoids data leakage:

1. **Game-level splitting**: We split PGN files (games) into train/val/test, *then* extract windows
   - **Why not window-level?** Windows from the same game are highly correlated. A model could memorize one game and seem to generalize well if we split at the window level.
   - **Correct approach**: If a game goes to the test set, all its windows stay in the test set.

2. **Stratified split**: Each player/engine class gets the same percentage in train/val/test (typically 80/10/10)
   - **Why?** Ensures the model sees balanced data and can fairly compare across classes.

3. **Balancing**: After extracting windows, we optionally downsample long games
   - **Why?** Long games produce many windows. Without balancing, a few prolific players could dominate the training data.

### Output: Three Lists of (tensor, label) Pairs

- `train_samples`: Windows for training (the model learns from these)
- `val_samples`: Windows for validation (used to select the best checkpoint)
- `test_samples`: Windows for testing (never seen by the model during training)

In [ ]:
from src.dataset import build_dataset, ChessDataset

if not all_pgns:
    print("Skipping dataset build because no PGN files are present.")
else:
    print("Building dataset from PGN files...")
    print("(This may take 1-5 minutes depending on the number of games)\n")
    
    # Scan data/raw/, extract games, split at game-level, then create windows
    train_samples, val_samples, test_samples = build_dataset(data_dir="data/raw")
    
    print("\n=== Dataset Summary ===")
    print(f"Train windows:  {len(train_samples):6d}")
    print(f"Val windows:    {len(val_samples):6d}")
    print(f"Test windows:   {len(test_samples):6d}")
    print(f"Total:          {len(train_samples) + len(val_samples) + len(test_samples):6d}")
    print("\nEach window is a (15, 34, 8, 8) tensor: 15 move states, 34 channels.")
    
    # Wrap samples in a PyTorch Dataset for easy batch loading
    train_dataset = ChessDataset(train_samples)
    print(f"\nTrain dataset created with {len(train_dataset)} windows.")
    print("Ready to pass to DataLoader for batching.")


Usable game counts:
  anand       : 3626
  botvinnik   : 774
  carlsen     : 7046
  fischer     : 729
  karpov      : 2964
  kasparov    : 1820
  petrosian   : 1316
  stockfish   : 998
  tal         : 1801
Using 729 games for anand
Using 729 games for botvinnik
Using 729 games for carlsen
Using 729 games for fischer
Using 729 games for karpov
Using 729 games for kasparov
Using 729 games for petrosian
Using 729 games for stockfish
Using 729 games for tal

Balanced usable game counts:
  anand       : 729
  botvinnik   : 729
  carlsen     : 729
  fischer     : 729
  karpov      : 729
  kasparov    : 729
  petrosian   : 729
  stockfish   : 729
  tal         : 729

Train game counts:
  anand       : 583
  botvinnik   : 583
  carlsen     : 583
  fischer     : 583
  karpov      : 583
  kasparov    : 583
  petrosian   : 583
  stockfish   : 583
  tal         : 583

Val game counts:
  anand       : 72
  botvinnik   : 72
  carlsen     : 72
  fischer     : 72
  karpov      : 72
  kasparov    : 72

## 6. Create the Neural Network

### Two Model Architectures

We compare two approaches:

#### ModelA: Baseline (CNN + Mean Pooling)

1. **Move Encoder (CNN)**: Each 34-channel move is processed independently
   - 3 convolutional layers: 34 → 64 → 128 → 256 channels
   - Each layer: Conv2d(3×3), BatchNorm, ReLU
   - Global average pooling → single 256-d vector per move

2. **Temporal Aggregation**: Average the move vectors across the window
   - Simple mean pooling over 15 moves → one game vector
   - Assumes all moves are equally important (no temporal structure)

3. **Classification**: Linear layer from 256 to num_classes
   - Outputs logits, softmax during training

**Intuition**: "Average the visual features; style is the mean of move patterns."

#### ModelC: Main Model (CNN + Transformer)

1. **Move Encoder (Residual CNN with Squeeze-Excitation)**:
   - 64-channel stem
   - 6 residual blocks with channel attention (squeeze-and-excitation gates)
   - Produces 320-dimensional move embeddings
   - More expressive than ModelA; can learn nonlinear move representations

2. **Positional Encoding**: Fixed sinusoidal encoding
   - Tells the model the position (1st move, 2nd move, ..., 15th move)
   - Essential so the model doesn't treat move order as irrelevant

3. **Transformer Encoder**: 12 layers, 8 attention heads, 1024 hidden dimension
   - Self-attention allows the model to relate moves across the window
   - "Move 3 was a rook sacrifice; move 12 was the payoff" ← requires long-range reasoning
   - Mean-pools the sequence output

4. **Embedding Head**: 512-dimensional bottleneck with LayerNorm and GELU
   - Converts transformer output to a compact style embedding

5. **Classifier**: Linear layer from 512 to num_classes

**Intuition**: "Style emerges from move sequences, not independent moves. Use attention to capture the narrative."

### Which One?

The default is **ModelC** because it can capture temporal relationships. ModelA is useful as a sanity check or if computational budget is tight.

In [ ]:
import importlib
import torch
import src.model as model_module
import src.pipeline as pipeline_module

# Reload modules to pick up any recent edits
model_module = importlib.reload(model_module)
pipeline_module = importlib.reload(pipeline_module)

# Import the model classes and pipeline utilities
ModelA = model_module.ModelA
ModelC = model_module.ModelC
make_model = pipeline_module.make_model
MODEL_CHOICE = pipeline_module.MODEL_CHOICE

# Instantiate the selected model
num_classes = len(LABELS)
model = make_model(MODEL_CHOICE, num_classes)

# Test with a dummy batch: (batch_size=2, window_size=15, channels=34, height=8, width=8)
dummy_batch = torch.zeros(2, 15, 34, 8, 8)
with torch.no_grad():
    logits = model(dummy_batch)

print("\n=== Model Configuration ===")
print(f"Default model choice:  {MODEL_CHOICE} (options: 'a' for ModelA, 'c' for ModelC)")
print(f"Instantiated model:    {model.__class__.__name__}")
print(f"Available models:      {ModelA.__name__} (baseline), {ModelC.__name__} (main)")

print("\n=== Forward Pass Test ===")
print(f"Input shape:           {tuple(dummy_batch.shape)}")
print(f"  (batch_size=2, window_size=15, channels=34, board_height=8, board_width=8)")
print(f"Output logits shape:   {tuple(logits.shape)}")
print(f"  (batch_size=2, num_classes={num_classes})")
print(f"\nModel is ready to train!")

/Users/aadil/Chess-ML-Project/chess-style-fingerprint/src/model.py:177: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


Default model choice: c
Model class: ModelC
Imported baseline model: ModelA
Imported stylometry model: ModelC
Input shape: (2, 15, 34, 8, 8)
Output logits shape: (2, 9)


## 7. Train the Model

### Training Loop Overview

`train_model()` in `src/pipeline.py` handles the full training pipeline:

1. **Dataset Loading**: Build train/val/test splits (this was done in cell 5)
2. **DataLoader Creation**: Batch and shuffle the data
3. **Model Setup**: Move model to GPU (if available)
4. **Optimizer & Loss**: AdamW optimizer + CrossEntropyLoss
5. **Main Loop**: For each epoch:
   - **Training phase**: Forward pass, backprop, parameter update on train batches
   - **Validation phase**: Evaluate on val set (no gradients)
   - **Checkpoint**: Save model if validation accuracy improved
6. **Early Stopping**: Stop if no improvement for 5 epochs (configurable)

### Key Hyperparameters

- `epochs`: Maximum number of training iterations (1 epoch = 1 pass through all train data)
- `batch_size`: How many windows per gradient step (larger batch = more stable but slower)
- `lr`: Learning rate for AdamW (controls step size)
- `weight_decay`: L2 regularization (prevents overfitting)
- `label_smoothing`: Softens targets (e.g., 0.9 instead of 1.0 for correct class)

### Note on Runtime

For this tutorial, we use `epochs=3` and `max_games_per_label=200` to keep the notebook interactive. For a real project, increase to `epochs=30` and use all available games.

In [ ]:
from src.pipeline import train_model

if not all_pgns:
    print("Skipping training because no PGN files are present.")
else:
    print("Starting training...\n")
    
    # Call the full training pipeline
    # Returns: trained model, test DataLoader, training history dict, torch device
    model, test_loader, history, device = train_model(
        data_dir="data/raw",
        epochs=3,  # Quick run for demo; use 30 for real training
        batch_size=64,  # Larger batch = more stable gradients, but slower
        max_games_per_label=200,  # Demo: limit to 200 games per class
        # max_games_per_label=None,  # Production: use all available games
        save_path="best_model.pt",
    )
    
    print("\n=== Training Complete ===")
    print(f"Training history keys: {list(history.keys())}")
    print(f"  - train_loss: training loss per epoch")
    print(f"  - val_loss: validation loss per epoch")
    print(f"  - val_acc: validation accuracy per epoch")
    print(f"\nDevice used: {device}")
    print(f"Best checkpoint saved to: best_model.pt")


Usable game counts:
  anand       : 3626
  botvinnik   : 774
  carlsen     : 7046
  fischer     : 729
  karpov      : 2964
  kasparov    : 1820
  petrosian   : 1316
  stockfish   : 998
  tal         : 1801
Using 200 games for anand
Using 200 games for botvinnik
Using 200 games for carlsen
Using 200 games for fischer
Using 200 games for karpov
Using 200 games for kasparov
Using 200 games for petrosian
Using 200 games for stockfish
Using 200 games for tal

Balanced usable game counts:
  anand       : 200
  botvinnik   : 200
  carlsen     : 200
  fischer     : 200
  karpov      : 200
  kasparov    : 200
  petrosian   : 200
  stockfish   : 200
  tal         : 200

Train game counts:
  anand       : 160
  botvinnik   : 160
  carlsen     : 160
  fischer     : 160
  karpov      : 160
  kasparov    : 160
  petrosian   : 160
  stockfish   : 160
  tal         : 160

Val game counts:
  anand       : 20
  botvinnik   : 20
  carlsen     : 20
  fischer     : 20
  karpov      : 20
  kasparov    : 20

## 8. Evaluate on the Test Split

### Why Test After Training?

- **Validation set** was used during training to pick the best checkpoint
- **Test set** was never seen by the model and gives an unbiased accuracy estimate
- **Test accuracy** is what we report; it reflects real-world performance

### Metrics Reported

1. **Overall test accuracy**: Percentage of windows correctly classified
2. **Per-class accuracy**: Accuracy for each player/engine (shows which classes are easier)
3. **Confusion matrix**: Most common misclassifications
   - If the model confuses Kasparov with Fischer often, they might have similar styles
   - If it's confused with Stockfish, it might be overfitting to tactical patterns

In [ ]:
from src.pipeline import test_predictions

if "model" not in globals() or "test_loader" not in globals() or "device" not in globals():
    print("Run the training cell (cell 7) before evaluating test predictions.")
else:
    print("Running inference on test set...\n")
    
    # Evaluate the model on the test split
    # Returns: true labels and predicted labels for all test windows
    y_true, y_pred = test_predictions(model, test_loader, device)
    
    print(f"Test evaluation complete.")
    print(f"Total predictions made: {len(y_pred)} windows")
    accuracy = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_pred) * 100
    print(f"Accuracy: {accuracy:.1f}%")

Test accuracy: 17.37%
Per-class accuracy:
  tal         :   0.0% (0/71)
  petrosian   :  46.5% (33/71)
  karpov      :  14.1% (10/71)
  fischer     :   0.0% (0/71)
  kasparov    :  54.9% (39/71)
  carlsen     :  31.0% (22/71)
  anand       :   4.2% (3/71)
  botvinnik   :   0.0% (0/71)
  stockfish   :   5.6% (4/71)
Top confusions:
  true=botvinnik    pred=carlsen      count=33
  true=carlsen      pred=petrosian    count=29
  true=stockfish    pred=carlsen      count=28
  true=tal          pred=kasparov     count=27
  true=karpov       pred=kasparov     count=26
  true=fischer      pred=kasparov     count=25
  true=anand        pred=kasparov     count=24
  true=botvinnik    pred=petrosian    count=23
  true=stockfish    pred=petrosian    count=23
  true=anand        pred=carlsen      count=20
Predictions made: 639


## 9. Save Training History

### Why Export as CSV?

The `history` dict contains training curves:
- `train_loss`: How well the model fit the training data (should decrease)
- `val_loss`: How well it generalized (should decrease, but may plateau/increase if overfitting)
- `val_acc`: Validation accuracy (should increase)

Exporting to CSV makes it easy to:
- **Plot in external tools** (Excel, Python, etc.)
- **Compare experiments**: Run training twice with different hyperparameters, plot both CSVs side-by-side
- **Reproducibility**: Archive the curves alongside the model checkpoint

In [ ]:
from src.pipeline import export_history_csv

if "history" not in globals():
    print("Run the training cell (cell 7) before exporting history.")
else:
    print("Exporting training history to CSV...")
    export_history_csv(history, out_path="training_history.csv")
    print("✓ Saved to: training_history.csv")
    print("\nYou can now plot this file to visualize:")
    print("  - Training loss (should decrease)")
    print("  - Validation loss (should decrease, may plateau)")
    print("  - Validation accuracy (should increase)")

Saved: training_history.csv


## 10. Run Inference on a PGN

### Using a Trained Model on New Data

Once trained, the model can predict style on any PGN file:

1. **Load checkpoint**: Restore model weights from `best_model.pt`
2. **Extract windows**: Convert the PGN into overlapping move windows (same as training)
3. **Batch predict**: Run all windows through the model
4. **Average probabilities**: Softmax outputs are averaged across windows
   - Window 1: [0.8, 0.1, 0.1] (predicts class 0)
   - Window 2: [0.7, 0.2, 0.1] (also predicts class 0)
   - Average: [0.75, 0.15, 0.1] ← final style fingerprint
5. **Rank classes**: Print in descending probability order

### Why Average Probabilities?

Each window is an independent prediction, but the game as a whole might have mixed signals (e.g., an engine-like middle game but tactical endgame). Averaging smooths out noise and gives a game-level estimate.

In [ ]:
from src.pipeline import infer_style_distribution

# Set this to a PGN file in your data directory
# Example: inference_pgn = "data/raw/tal/example_game.pgn"
inference_pgn = ""  

if not Path("best_model.pt").exists():
    print("✗ No best_model.pt checkpoint found.")
    print("   Run the training cell (cell 7) first.")
elif not inference_pgn:
    print("Set inference_pgn to a PGN file path.")
    print("Example: inference_pgn = \"data/raw/tal/example_game.pgn\"")
else:
    print(f"Inferring style for: {inference_pgn}\n")
    
    # Load the trained model and predict the style distribution
    ranking = infer_style_distribution(inference_pgn, save_path="best_model.pt")
    
    print(f"\n=== Style Ranking ===")
    print(f"Top 3 most likely styles:")
    for i, (name, prob) in enumerate(ranking[:3], 1):
        print(f"  {i}. {name.capitalize():12s} {prob*100:5.1f}%")

Set inference_pgn to a PGN path before running inference.


## 11. Full Pipeline Shortcut (Optional)

### One-Step Automation

`run_full_pipeline()` is a convenience function that does everything in one call:

1. Train the model
2. Evaluate on test set
3. Export training history
4. (Optionally) run inference on a PGN

This is useful for **quick experiments** once you understand the flow.

### When to Use

- **Development**: Use individual cells (1–10) to understand each step and debug
- **Production**: Use the full pipeline for reproducible training runs

The individual cells give you transparency; the full pipeline saves time.

In [ ]:
from src.pipeline import run_full_pipeline

# Uncomment the line below to run the complete pipeline end-to-end.
# This will train, evaluate, save history, and optionally run inference.
# For a dry run, set inference_pgn to an empty string.

# model, history = run_full_pipeline(inference_pgn="")

print("To run the full pipeline, uncomment the line above.")
print("Example with inference:")
print('  model, history = run_full_pipeline(inference_pgn="data/raw/tal/example_game.pgn")')